# Session 3 — Linear Regression

*Machine Learning · Grau en Enginyeria Informàtica · Universitat de Barcelona · 2026-27*

> **Today's question.** You have **$50k** to spend on advertising in one market. Where do you put it?
> **A)** all on TV · **B)** all on radio · **C)** half and half · **D)** newspaper
>
> Write your answer down. We come back to it in Section 3.

**Contents**

1. The best line — least squares and the normal equation
2. Several variables — what a coefficient means, and how sure we are
3. Qualitative predictors and interactions
4. Beyond the line — polynomial regression and overfitting
5. Regularization — Ridge and Lasso
6. Wrap-up
7. Practice at home — medical insurance charges

**How to use this notebook.** In class we run some cells live. A cell preceded by **Predict first** is a
question: commit to an answer *before* running it.

**Notation** (same as the slides)

| Symbol | Meaning |
| --- | --- |
| $n$ | number of observations |
| $p$ | number of features |
| $x_i \in \mathbb{R}^p,\ y_i$ | features and target of observation $i$ |
| $X \in \mathbb{R}^{n \times (p+1)}$ | design matrix (first column of ones) |
| $\beta = (\beta_0, \dots, \beta_p)$ | coefficients; $\hat\beta$ is their estimate |
| $\hat y_i = \hat\beta_0 + \sum_j \hat\beta_j x_{ij}$ | prediction |
| $\mathrm{RSS} = \sum_i (y_i - \hat y_i)^2$ | residual sum of squares |

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path
from scipy.optimize import minimize
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, KFold, RepeatedKFold, cross_val_score, validation_curve

SEED = 42  # every random step uses this seed: re-running gives identical numbers

# Plot style: one colour per role, used consistently through the notebook
BLUE, ORANGE, GREEN, YELLOW, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a8a85"
plt.rcParams.update({
    "figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, GREEN, YELLOW]),
})

print(f"numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")


def rmse(y_true, y_pred):
    """Root mean squared error, in the units of y."""
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))


def load_csv(local_path, url, **kwargs):
    """Read the course copy of a dataset; download it from `url` if it is not there."""
    path = Path(local_path)
    return pd.read_csv(path if path.exists() else url, **kwargs)

We load the two datasets now, so a download problem shows up at the start and not in the middle of the class.

In [ ]:
# Advertising (ISLR): sales of a product in 200 markets and its advertising budget
# per medium. Budgets in thousands of dollars, sales in thousands of units.
ads = load_csv("dataset/advertising/Advertising.csv",
               "https://raw.githubusercontent.com/justmarkham/scikit-learn-videos/master/data/Advertising.csv",
               index_col=0)

# Medical insurance: yearly charges billed to 1,338 insured people (used in Sections 3 and 7)
ins = load_csv("dataset/regression_healthcare/datasets_13720_18513_insurance.csv",
               "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv")

print("Advertising:", ads.shape, "| Insurance:", ins.shape)
ads.head()

## 1. The best line

### 1.1 Fit a line by eye

Twenty observations generated from a line we keep hidden, plus noise.

In [ ]:
rng = np.random.default_rng(SEED)
n = 20
x = rng.uniform(0, 2, n)
y = 4 + 3 * x + rng.normal(0, 1, n)   # hidden truth: beta0 = 4, beta1 = 3, noise sd = 1

plt.scatter(x, y, color=BLUE)
plt.xlabel("$x$"); plt.ylabel("$y$"); plt.title("20 observations: which line fits best?")
plt.show()

**In pairs (1 min).** Propose an intercept $b_0$ and a slope $b_1$. Type the class proposals in the cell below.

How do we decide which proposal is best? We need a single number per line.

In [ ]:
def rss(b0, b1, x=x, y=y):
    """Residual sum of squares of the line y = b0 + b1 * x."""
    return float(np.sum((y - (b0 + b1 * x)) ** 2))

proposals = {           # <- replace with the class proposals: "name": (b0, b1)
    "Pair A": (3.0, 3.5),
    "Pair B": (5.0, 2.0),
    "Pair C": (4.0, 3.0),
    "Pair D": (2.5, 4.0),
}

# Let the computer search for the (b0, b1) with the smallest RSS
best = minimize(lambda b: rss(*b), x0=[0.0, 0.0]).x
proposals["Least squares"] = tuple(best)

table = pd.DataFrame(proposals, index=["b0", "b1"]).T
table["RSS"] = [rss(b0, b1) for b0, b1 in proposals.values()]
table.sort_values("RSS").round(3)

In [ ]:
grid = np.linspace(0, 2, 50)
plt.scatter(x, y, color=BLUE, zorder=3)
for name, (b0, b1) in proposals.items():
    if name != "Least squares":
        plt.plot(grid, b0 + b1 * grid, color=GREY, lw=1, alpha=0.7)
b0, b1 = proposals["Least squares"]
plt.plot(grid, b0 + b1 * grid, color=ORANGE, lw=2, label="least squares")
plt.vlines(x, y, b0 + b1 * x, color=ORANGE, lw=1, ls=":", label="residuals")
plt.xlabel("$x$"); plt.ylabel("$y$"); plt.legend(); plt.title("Class proposals (grey) vs least squares")
plt.show()

### 1.2 Why squares?

Least squares minimises $\sum_i (y_i - \hat y_i)^2$. We could minimise $\sum_i |y_i - \hat y_i|$ instead.

**Predict first.** We add **one** outlier to the data. Which line moves more: the one fitted with squared
loss or the one fitted with absolute loss?

In [ ]:
x_out = np.append(x, 1.9)
y_out = np.append(y, 0.0)      # one observation far below the others

def fit(loss, x, y):
    """Fit y = b0 + b1 x by minimising sum(loss(residuals)) numerically."""
    objective = lambda b: np.sum(loss(y - b[0] - b[1] * x))
    return minimize(objective, x0=[0.0, 0.0], method="Nelder-Mead").x

squared, absolute = np.square, np.abs
fits = {
    "squared, clean data":     fit(squared, x, y),
    "squared, with outlier":   fit(squared, x_out, y_out),
    "absolute, clean data":    fit(absolute, x, y),
    "absolute, with outlier":  fit(absolute, x_out, y_out),
}
display(pd.DataFrame(fits, index=["b0", "b1"]).T.round(2))

plt.scatter(x, y, color=GREY)
plt.scatter([1.9], [0.0], color="black", marker="x", s=80, label="outlier")
for (name, (b0, b1)), color, ls in zip(fits.items(), [ORANGE, ORANGE, GREEN, GREEN], ["-", "--", "-", "--"]):
    plt.plot(grid, b0 + b1 * grid, color=color, ls=ls, lw=2, label=name)
plt.legend(fontsize=9); plt.title("Squared loss is pulled by the outlier; absolute loss much less")
plt.show()

Squaring makes large residuals dominate. The same happens with metrics: RMSE punishes a few large errors more
than MAE. We keep squared loss because it has a closed-form solution — next cell.

### 1.3 The normal equation

Write all $n$ predictions at once as $\hat y = X\beta$. Then

$$
\mathrm{RSS}(\beta) = \lVert y - X\beta \rVert_2^2, \qquad
\nabla_\beta \mathrm{RSS} = -2X^\top (y - X\beta) = 0
\;\Longrightarrow\; X^\top X\,\hat\beta = X^\top y .
$$

If the columns of $X$ are linearly independent, $X^\top X$ is invertible and the solution is unique:
$\hat\beta = (X^\top X)^{-1}X^\top y$. In code we **solve** the system; we never compute the inverse
(it is slower and numerically less stable).

In [ ]:
X = np.column_stack([np.ones(n), x])                 # design matrix: column of ones + feature

beta_solve = np.linalg.solve(X.T @ X, X.T @ y)       # solve the normal equations
beta_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)   # least squares via SVD (what scikit-learn uses)
lin = LinearRegression().fit(x.reshape(-1, 1), y)    # scikit-learn expects a 2-D X

pd.DataFrame({
    "solve(XᵀX, Xᵀy)": beta_solve,
    "lstsq": beta_lstsq,
    "LinearRegression": [lin.intercept_, lin.coef_[0]],
}, index=["beta0", "beta1"]).round(4)

All three agree. Solving costs about $O(np^2 + p^3)$ operations: fine for $p = 10$, not for $p = 10^5$.
That is the motivation for gradient descent (next session).

### 1.4 Units

**Predict first.** We fit `Sales ~ TV` with TV measured in **dollars** instead of thousands of dollars.
What happens to $\hat\beta_{TV}$, to the predictions and to $R^2$?

In [ ]:
for unit, factor in [("thousands of $", 1), ("dollars", 1000)]:
    X_tv = ads[["TV"]] * factor
    m = LinearRegression().fit(X_tv, ads["Sales"])
    print(f"TV in {unit:15s} beta_TV = {m.coef_[0]:.6f}   R² = {m.score(X_tv, ads['Sales']):.4f}")

The coefficient is divided by 1000; predictions and $R^2$ do not change. Remember this for Section 5.

## 2. Several variables

### 2.1 The newspaper mystery

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, medium in zip(axes, ["TV", "Radio", "Newspaper"]):
    ax.scatter(ads[medium], ads["Sales"], s=12, alpha=0.6, color=BLUE)
    ax.set_xlabel(f"{medium} budget (k$)")
axes[0].set_ylabel("Sales (k units)")
plt.show()

In [ ]:
def coef_table(model):
    """Coefficient, standard error, 95% CI and p-value of a fitted statsmodels OLS."""
    ci = model.conf_int()
    return pd.DataFrame({"coef": model.params, "std err": model.bse,
                         "CI 2.5%": ci[0], "CI 97.5%": ci[1], "p-value": model.pvalues}).round(4)

m_news = smf.ols("Sales ~ Newspaper", data=ads).fit()
print(f"R² = {m_news.rsquared:.3f}")
coef_table(m_news)

Alone, newspaper looks useful: positive coefficient, p-value ≈ 0.001.

**Predict first.** We add TV and Radio to the model. The Newspaper coefficient will be…
**A)** similar and significant · **B)** close to 0, not significant · **C)** negative and significant · **D)** larger

In [ ]:
m_all = smf.ols("Sales ~ TV + Radio + Newspaper", data=ads).fit()
print(f"R² = {m_all.rsquared:.3f}")
coef_table(m_all)

In [ ]:
ads[["TV", "Radio", "Newspaper"]].corr().round(2)

Markets that spend more on radio also spend more on newspapers (correlation 0.35). Alone, newspaper gets the
credit for radio. In multiple regression

> $\beta_j$ = expected change in $Y$ when $X_j$ increases by one unit **and the other features stay fixed**.

With observational data this is an association, **not** a causal effect.

### 2.2 How sure are we? Standard errors and confidence intervals

$\hat\beta$ depends on the sample we happened to observe. Its **standard error** is how much it would vary
across samples. A 95% confidence interval is roughly $\hat\beta \pm 2\,\mathrm{SE}(\hat\beta)$, and the
**p-value** answers: *if the true $\beta$ were 0, how rare would an estimate this far from 0 be?*

What does "95%" refer to? We simulate it: draw 100 samples from a model whose true slope we know, and build
a 95% interval from each.

In [ ]:
rng = np.random.default_rng(SEED)
true_b0, true_b1, noise_sd, n_sim = 7.0, 0.05, 3.0, 200

intervals = []
for _ in range(100):
    tv = rng.uniform(0, 300, n_sim)
    sales = true_b0 + true_b1 * tv + rng.normal(0, noise_sd, n_sim)
    fit_sim = sm.OLS(sales, sm.add_constant(tv)).fit()
    intervals.append(fit_sim.conf_int()[1])          # 95% CI for the slope
intervals = np.array(intervals)
covers = (intervals[:, 0] <= true_b1) & (true_b1 <= intervals[:, 1])

plt.figure(figsize=(7, 5))
for k, (lo, hi) in enumerate(intervals):
    plt.plot([lo, hi], [k, k], color=BLUE if covers[k] else ORANGE, lw=1.5)
plt.axvline(true_b1, color="black", lw=1)
plt.text(true_b1, 101, f" true slope = {true_b1}", va="bottom")
plt.ylim(-1, 106)
plt.xlabel("95% CI for the slope"); plt.ylabel("sample")
plt.title(f"{covers.sum()} of 100 intervals contain the true slope (orange: missed)")
plt.show()

The **procedure** captures the true value in about 95% of samples. A single computed interval either contains
$\beta$ or it does not; it is not "95% probable" that $\beta$ lies in it.

## 3. Qualitative predictors and interactions

### 3.1 Dummy variables

`region` in the insurance data has 4 levels.

**Think–pair–share.** With an intercept in the model, how many dummy (0/1) columns do we need for `region`?
What goes wrong if we use 4?

In [ ]:
dummies = pd.get_dummies(ins["region"], dtype=float)           # one 0/1 column per region
X_four = np.column_stack([np.ones(len(ins)), dummies])          # intercept + 4 dummies
X_three = np.column_stack([np.ones(len(ins)), dummies.iloc[:, 1:]])  # drop the first level

print("intercept + 4 dummies: columns =", X_four.shape[1], " rank =", np.linalg.matrix_rank(X_four))
print("intercept + 3 dummies: columns =", X_three.shape[1], " rank =", np.linalg.matrix_rank(X_three))

The 4 dummies add up to the column of ones, so $X$ loses rank and $X^\top X$ is singular: the normal equation
has no unique solution (the *dummy variable trap*). With 3 dummies, the dropped level is the **baseline**.

In [ ]:
m_region = smf.ols("charges ~ C(region)", data=ins).fit()   # statsmodels drops the first level itself
display(coef_table(m_region))
ins.groupby("region")["charges"].mean().round(1)

The intercept is the mean charge in the baseline region (northeast); each coefficient is the difference between
that region and the baseline. In scikit-learn the same encoding is `OneHotEncoder(drop="first")`.

### 3.2 Interactions

The additive model assumes the effect of TV does not depend on how much we spend on radio. Let us allow it to:

$$
\text{Sales} = \beta_0 + \beta_1\,\text{TV} + \beta_2\,\text{Radio} + \beta_3\,(\text{TV}\times\text{Radio}) + \varepsilon
= \beta_0 + (\beta_1 + \beta_3\,\text{Radio})\,\text{TV} + \beta_2\,\text{Radio} + \varepsilon
$$

In [ ]:
m_add = smf.ols("Sales ~ TV + Radio", data=ads).fit()
m_int = smf.ols("Sales ~ TV * Radio", data=ads).fit()     # TV * Radio = TV + Radio + TV:Radio
print(f"R² additive = {m_add.rsquared:.3f}   R² with interaction = {m_int.rsquared:.3f}")
coef_table(m_int)

**In pairs (5 min).** With the coefficients above, compute predicted sales for three ways of spending $50k:
all on TV, all on radio, half and half. Then run the cell.

In [ ]:
budgets = pd.DataFrame({"TV": [50, 0, 25], "Radio": [0, 50, 25]},
                       index=["All on TV", "All on radio", "Half and half"])
budgets["Additive model"] = m_add.predict(budgets)
budgets["Interaction model"] = m_int.predict(budgets)
budgets.round(2)

**Back to today's question.** The additive model says *all on radio*; the model with interaction says
*half and half*. The model you choose changes the decision.

**Discuss.** Would you trust the model's recommendation for a $200k radio budget? Check the range of the data:

In [ ]:
ads[["TV", "Radio"]].agg(["min", "max"])

Radio never exceeds about $50k in the data: predicting far outside the observed range is extrapolation.
*Hierarchy principle:* if a model includes an interaction, it also keeps the main effects, even if their
p-values are large.

## 4. Beyond the line: polynomial regression

A polynomial in $x$ is still **linear in the parameters**:
$\hat y = \beta_0 + \beta_1 x + \beta_2 x^2 + \dots + \beta_d x^d$.
We just add the columns $x^2, \dots, x^d$ to $X$ and use the same least squares.

We take 20 training points from $y = \sin(2\pi x) + \varepsilon$ and keep 500 new points aside to measure the
error on data the model has not seen.

In [ ]:
rng = np.random.default_rng(SEED)
f_true = lambda x: np.sin(2 * np.pi * x)
x_tr = rng.uniform(0, 1, 20)
y_tr = f_true(x_tr) + rng.normal(0, 0.25, 20)
x_te = rng.uniform(0, 1, 500)
y_te = f_true(x_te) + rng.normal(0, 0.25, 500)

def poly_model(degree):
    return make_pipeline(PolynomialFeatures(degree), LinearRegression())

**Predict first.** Which degree will have the lowest error on the 500 new points: 1, 3 or 15?

In [ ]:
grid = np.linspace(0, 1, 400)
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5), sharey=True)
for ax, degree in zip(axes, [1, 3, 9, 15]):
    model = poly_model(degree).fit(x_tr.reshape(-1, 1), y_tr)
    ax.plot(grid, f_true(grid), color=GREY, ls="--", lw=1, label="true function")
    ax.plot(grid, model.predict(grid.reshape(-1, 1)), color=ORANGE, lw=2, label="fit")
    ax.scatter(x_tr, y_tr, color=BLUE, s=20, zorder=3, label="training data")
    train_err = rmse(y_tr, model.predict(x_tr.reshape(-1, 1)))
    test_err = rmse(y_te, model.predict(x_te.reshape(-1, 1)))
    ax.set_title(f"degree {degree}\ntrain RMSE {train_err:.3f} · test RMSE {test_err:.3f}", fontsize=10)
    ax.set_ylim(-2, 2)
axes[0].legend(fontsize=8, loc="lower left")
plt.show()

In [ ]:
degrees = range(1, 16)
train_rmse, test_rmse = [], []
for degree in degrees:
    model = poly_model(degree).fit(x_tr.reshape(-1, 1), y_tr)
    train_rmse.append(rmse(y_tr, model.predict(x_tr.reshape(-1, 1))))
    test_rmse.append(rmse(y_te, model.predict(x_te.reshape(-1, 1))))

plt.plot(degrees, train_rmse, "o-", color=BLUE, label="training data")
plt.plot(degrees, test_rmse, "o-", color=ORANGE, label="new data")
plt.xlabel("polynomial degree"); plt.ylabel("RMSE"); plt.legend()
plt.title("Training error always goes down; error on new data does not")
plt.show()

More flexibility always lowers the training error. The error on new data first drops and then, irregularly,
goes up again: too simple a model misses the pattern (*underfitting*), too flexible a model fits the noise
(*overfitting*). With only 20 points the curve is jumpy, but degree 3 beats degree 15 clearly. The training error
alone cannot choose the degree.

## 5. Regularization: Ridge and Lasso

Keep a flexible model, but penalise large coefficients:

$$
\text{Ridge: } \min_\beta\ \mathrm{RSS}(\beta) + \lambda \sum_{j=1}^{p} \beta_j^2
\qquad\qquad
\text{Lasso: } \min_\beta\ \mathrm{RSS}(\beta) + \lambda \sum_{j=1}^{p} |\beta_j|
$$

The intercept $\beta_0$ is not penalised. $\lambda \ge 0$ controls the trade-off: $\lambda = 0$ is ordinary least
squares. In scikit-learn $\lambda$ is called `alpha` (and `Lasso` divides the RSS by $2n$, so `alpha` values are
not comparable between `Ridge` and `Lasso`). *Elastic Net* combines both penalties (`ElasticNet`).

### 5.1 Units, again

**Predict first.** In Section 1.4 changing the units of TV did not change OLS predictions. Now we use
`Ridge()` with its default `alpha=1` and measure TV in **millions** of dollars. Do the predictions change?

In [ ]:
X_ads, y_ads = ads[["TV", "Radio", "Newspaper"]], ads["Sales"]
X_millions = X_ads.assign(TV=X_ads["TV"] / 1000)        # same data, TV in millions of $

models = {
    "OLS": LinearRegression(),
    "Ridge(alpha=1)": Ridge(alpha=1.0),
    "StandardScaler + Ridge(alpha=1)": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
}
rows = {}
for name, model in models.items():
    pred_k = model.fit(X_ads, y_ads).predict(X_ads)
    pred_m = model.fit(X_millions, y_ads).predict(X_millions)
    rows[name] = {"RMSE, TV in k$": rmse(y_ads, pred_k),
                  "RMSE, TV in M$": rmse(y_ads, pred_m),
                  "max change in a prediction": np.abs(pred_k - pred_m).max()}
pd.DataFrame(rows).T.round(3)

In millions, the TV coefficient becomes 1000 times larger, so the penalty shrinks it hard and the fit gets worse.
The penalty treats all coefficients alike, so **features must be on the same scale**: always put a
`StandardScaler` before `Ridge`/`Lasso`, inside a `Pipeline` (so it is fitted on training data only).

### 5.2 Coefficient paths

**Predict first.** As $\lambda$ grows, which Advertising coefficient will Lasso set to exactly 0 first?

In [ ]:
def coef_path(estimator, alphas):
    """Standardised coefficients of estimator(alpha) for each alpha."""
    return np.array([make_pipeline(StandardScaler(), estimator(alpha=a)).fit(X_ads, y_ads)[-1].coef_
                     for a in alphas])

ridge_alphas, lasso_alphas = np.logspace(-1, 5, 100), np.logspace(-3, 1, 100)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (title, estimator, alphas) in zip(axes, [("Ridge", Ridge, ridge_alphas), ("Lasso", Lasso, lasso_alphas)]):
    for coefs, name, color in zip(coef_path(estimator, alphas).T, X_ads.columns, [BLUE, ORANGE, GREEN]):
        ax.plot(alphas, coefs, color=color, lw=2, label=name)
    ax.set_xscale("log"); ax.axhline(0, color="black", lw=0.8)
    ax.set_xlabel("alpha (λ)"); ax.set_title(title)
axes[0].set_ylabel("standardised coefficient"); axes[0].legend()
plt.show()

Ridge shrinks every coefficient towards 0 but never to exactly 0. Lasso sets coefficients to exactly 0 one by one:
it also **selects features**. Newspaper goes first, as Section 2 suggested.

### 5.3 Choosing λ with validation data

Back to the degree-15 polynomial from Section 4, which overfitted. We add a Ridge penalty and choose `alpha`
with cross-validation on the 20 training points only (5 folds, repeated 10 times with different shuffles,
because with 4 points per fold a single split is very noisy). The 500 new points are used once, at the end.

In [ ]:
ridge_poly = make_pipeline(PolynomialFeatures(15, include_bias=False), StandardScaler(), Ridge())
alphas = np.logspace(-8, 2, 41)
cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=SEED)

train_scores, cv_scores = validation_curve(
    ridge_poly, x_tr.reshape(-1, 1), y_tr, param_name="ridge__alpha", param_range=alphas,
    cv=cv, scoring="neg_root_mean_squared_error")
train_err, cv_err = -train_scores.mean(axis=1), -cv_scores.mean(axis=1)
best_alpha = alphas[cv_err.argmin()]

plt.plot(alphas, train_err, "o-", color=BLUE, ms=4, label="training folds")
plt.plot(alphas, cv_err, "o-", color=ORANGE, ms=4, label="validation folds")
plt.axvline(best_alpha, color=GREY, ls="--", label=f"best alpha ≈ {best_alpha:.3g}")
plt.xscale("log"); plt.yscale("log")
plt.yticks([0.15, 0.25, 0.5, 1, 2], ["0.15", "0.25", "0.5", "1", "2"])
plt.gca().yaxis.set_minor_formatter(plt.NullFormatter())
plt.xlabel("alpha (λ)"); plt.ylabel("RMSE (cross-validation)"); plt.legend()
plt.title("Degree-15 polynomial + Ridge: choose alpha on validation data")
plt.show()

In [ ]:
final = ridge_poly.set_params(ridge__alpha=best_alpha).fit(x_tr.reshape(-1, 1), y_tr)
plain15 = poly_model(15).fit(x_tr.reshape(-1, 1), y_tr)
plain3 = poly_model(3).fit(x_tr.reshape(-1, 1), y_tr)

pd.Series({
    "degree 3, no penalty": rmse(y_te, plain3.predict(x_te.reshape(-1, 1))),
    "degree 15, no penalty": rmse(y_te, plain15.predict(x_te.reshape(-1, 1))),
    f"degree 15 + Ridge (alpha={best_alpha:.3g})": rmse(y_te, final.predict(x_te.reshape(-1, 1))),
}, name="RMSE on the 500 new points").round(3)

With the right penalty, the degree-15 model generalises as well as the degree-3 one or better: regularization is
another way to control complexity, and λ is chosen on validation data, never on the test set.

## 6. Wrap-up

**Exit ticket**

1. Why don't we use the normal equation when $p = 100{,}000$?
2. In `Sales ~ TV * Radio`, what does the TV coefficient (≈ 0.019) mean?
3. A colleague reports $R^2 = 0.99$ on the training data with a degree-15 polynomial. What do you ask first?
4. What was the least clear point today?

**Key ideas**

- Least squares minimises the RSS; its solution solves $X^\top X\hat\beta = X^\top y$.
- In multiple regression, a coefficient is an effect *holding the other features fixed* — an association, not a cause.
- A categorical feature with $K$ levels needs $K-1$ dummies; interactions let the effect of one feature depend on another.
- Training error always falls with flexibility; judge models on data they have not seen.
- Ridge and Lasso penalise large coefficients: standardise first, choose λ by cross-validation. Lasso also selects features.

## 7. Practice at home: medical insurance charges

Predict the yearly medical `charges` of a person from `age`, `sex`, `bmi`, `children`, `smoker` and `region`.
Charges are very skewed, so we predict $\log(\text{charges})$ and report errors on that scale.

In [ ]:
ins_model = ins.assign(log_charges=np.log(ins["charges"]))
num_features = ["age", "bmi", "children"]
cat_features = ["sex", "smoker", "region"]
X_ins = ins_model[num_features + cat_features]
y_ins = ins_model["log_charges"]

**Task 1.** Split the data into 80% training and 20% test with `random_state=SEED`.
Build a preprocessing step with `make_column_transformer`: `StandardScaler` for the numerical features and
`OneHotEncoder(drop="first")` for the categorical ones.

In [ ]:
# TODO: split 80/20 with random_state=SEED
X_train, X_test, y_train, y_test = ...

# TODO: StandardScaler for num_features, OneHotEncoder(drop="first") for cat_features
preprocess = ...

**Task 2.** Compare `LinearRegression`, `Ridge(alpha=1)` and `Lasso(alpha=0.01)` with 5-fold cross-validation
on the training set (RMSE). Each model goes in a pipeline after `preprocess`.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
candidates = {
    "Linear regression": LinearRegression(),
    "Ridge (alpha=1)": Ridge(alpha=1.0),
    "Lasso (alpha=0.01)": Lasso(alpha=0.01),
}
for name, estimator in candidates.items():
    # TODO: build the pipeline and compute the cross-validated RMSE
    scores = ...
    print(f"{name:20s} CV RMSE = {-scores.mean():.4f} ± {scores.std():.4f}")

**Task 3.** Smoking may change the effect of BMI on charges. Add the interaction feature
`smoker_bmi = bmi × (smoker == "yes")` and check whether the cross-validated RMSE of the linear model improves.

In [ ]:
# TODO: add the interaction column to X_train and X_test, and to the numerical features
X_train_int = ...
X_test_int = ...
preprocess_int = ...

# TODO: cross-validated RMSE of LinearRegression with the interaction

**Task 4.** Pick your best model using the cross-validation results only. Fit it on the whole training set and
evaluate it **once** on the test set. Is the test RMSE close to the CV estimate?

In [ ]:
# TODO: fit the chosen pipeline on the training set and compute the test RMSE

**Question for Practice 1.** In the house price competition, what would change if you trained the model on
$\log(\text{Price})$ and converted the predictions back with $\exp$? Think about which errors each choice
penalises most.